# Del Video al Audio: Pipeline de Extracción con FFmpeg

En este notebook demostraremos el flujo completo para convertir un archivo de video a audio WAV:

1. **Visualizar** el video directamente en el notebook.
2. **Extraer** el audio usando nuestra función `helpers/video_utils.py` (sin librerías extra).
3. **Reproducir** el archivo `.wav` resultante.

In [1]:
import sys
import os
from pathlib import Path
from IPython.display import Video, Audio, display

# Nos aseguramos de que Python pueda encontrar la carpeta 'helpers'
# (necesario cuando el notebook se ejecuta desde un subdirectorio)
sys.path.insert(0, str(Path('.').resolve()))

from helpers.video_utils import extract_audio_from_video

print('✅ Importaciones exitosas.')

✅ Importaciones exitosas.


## Sección 1: Visualización del Video

Especificamos la ruta al video de entrada y lo mostramos directamente dentro del notebook usando `IPython.display.Video`.

> **👇 Cambia el valor de `VIDEO_PATH` a la ruta de tu propio video.**

In [2]:
# ─── CONFIGURACIÓN ─────────────────────────────────────────────────────────
# Cambia esta variable a la ruta de tu archivo de video.
VIDEO_PATH = r"data\video\quien-me-jalo-mi-cabello.mp4"

# Ruta donde se guardará el .wav extraído.
# Si la dejas en None, se usará el mismo directorio y nombre que el video.
OUTPUT_WAV = None
# ───────────────────────────────────────────────────────────────────────────

# Verificamos que el archivo exista antes de continuar
if not Path(VIDEO_PATH).exists():
    print(f"⚠️  Archivo no encontrado: '{VIDEO_PATH}'")
    print("   Actualiza la variable VIDEO_PATH con la ruta correcta.")
else:
    size_mb = Path(VIDEO_PATH).stat().st_size / 1024 / 1024
    print(f"📁 Video encontrado: {VIDEO_PATH} ({size_mb:.1f} MB)")

    # Mostramos el video embebido en el notebook
    # embed=True incrusta el video como base64 para que funcione sin servidor HTTP
    display(Video(VIDEO_PATH, embed=True, width=720))

📁 Video encontrado: data\video\quien-me-jalo-mi-cabello.mp4 (2.4 MB)


## Sección 2: Extracción del Audio

Usamos `extract_audio_from_video()` de nuestro módulo `helpers`. La función llama a `ffmpeg` internamente via `subprocess`, por lo que **no se necesita instalar ninguna librería adicional de Python**.

El audio se guardará como WAV **mono a 22 050 Hz**, formato ideal para análisis con `torchaudio`.

In [3]:
# Ejecutamos la extracción.
# La función imprime el progreso y retorna la ruta del archivo generado.
wav_path = extract_audio_from_video(
    video_path=VIDEO_PATH,
    output_wav_path=OUTPUT_WAV,
    sample_rate=22050,  # Hz — suficiente para ML/análisis de voz
    channels=1,         # Mono — reduce dimensionalidad, ideal para CNNs
)

print(f"\n📄 Archivo WAV generado en: {wav_path}")

⚙️  Extrayendo audio de: quien-me-jalo-mi-cabello.mp4
   Guardando en        : C:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\uag-deep-learning\term_02_unstructured_data\data\video\quien-me-jalo-mi-cabello.wav
✅ Audio extraído exitosamente (22050 Hz, Mono)

📄 Archivo WAV generado en: C:\Users\efrai\OneDrive\Desktop\EF\Education\Professor\DL classes\uag-deep-learning\term_02_unstructured_data\data\video\quien-me-jalo-mi-cabello.wav


## Sección 3: Reproducción del Audio Extraído

Cargamos el `.wav` con `torchaudio` para confirmar su forma como tensor, y lo reproducimos con el widget de audio de Jupyter.

Esta es exactamente la misma entrada que recibirá nuestro pipeline de espectrogramas del notebook `04_audio_to_spectrogram.ipynb`.

In [4]:
import torchaudio

# Cargamos el WAV como tensor de PyTorch
waveform, sample_rate = torchaudio.load(wav_path)

# Información del tensor resultante
duracion_s = waveform.shape[1] / sample_rate
print("──────────────────────────────────────────")
print("AUDIO EXTRAÍDO — RESUMEN DEL TENSOR")
print("──────────────────────────────────────────")
print(f"  Ruta         : {wav_path}")
print(f"  Shape tensor : {waveform.shape}  → [canales, muestras]")
print(f"  Sample Rate  : {sample_rate} Hz")
print(f"  Duración     : {duracion_s:.2f} segundos")
print("──────────────────────────────────────────")

# Reproducimos el audio directamente en el notebook
display(Audio(waveform.numpy()[0], rate=sample_rate))

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call 